# 🧩 Eino ADK 内置中间件一览

    Summarization（摘要）
        核心功能：当对话历史的 Token 数或消息数超过阈值时，自动调用 LLM 生成摘要，压缩上下文，防止超出模型窗口限制。
        关键配置：TriggerCondition（触发阈值，如 ContextTokens 或 ContextMessages）、PreserveUserMessages（保留最近用户消息）、Callback（摘要前后钩子）。

    ToolReduction（工具结果压缩）
        核心功能：分两阶段管理工具输出。截断：工具返回后截断超长内容并保存到文件；清理：模型调用前清理历史工具结果，释放 Token。
        关键配置：MaxLengthForTrunc（触发截断的长度阈值）、MaxTokensForClear（触发清理的 Token 阈值）。

    Filesystem（文件系统）
        核心功能：为 Agent 注入一组文件操作工具（ls、read_file、write_file、edit_file、glob、grep），可选注入 execute 工具执行命令。
        关键配置：Backend（必填，文件系统后端）、Shell / StreamingShell（可选，命令执行能力）、UseMultiModalRead（启用多模态读取）、各工具的 ToolConfig（自定义名称、描述或禁用）。

    Skill（技能）
        核心功能：动态加载和执行预定义的 Skill，支持 fork 和 isolate 两种上下文模式，允许通过 FrontMatter 指定 agent 和 model。
        关键配置：通常涉及 Skill 的定义目录、上下文模式选择。

    PlanTask（任务规划）
        核心功能：通过 BeforeAgent 钩子注入四个任务管理工具：TaskCreate、TaskGet、TaskUpdate、TaskList，让 Agent 能够创建、跟踪和管理结构化任务列表。
        关键配置：Backend（任务存储后端，需按 session 隔离）、BaseDir（任务文件存储目录）。

    ToolSearch（工具搜索）
        核心功能：当可用工具数量庞大时，通过语义搜索动态选择最相关的工具，避免将所有工具定义塞入上下文导致过载。
        关键配置：搜索后端、检索数量等。

    PatchToolCalls（修补工具调用）
        核心功能：扫描消息历史，为那些“有调用但缺少响应”的悬空工具调用自动插入占位符消息，确保消息历史的完整性，防止因中断导致后续对话报错。
        关键配置：基础用法通常无需额外配置，patchtoolcalls.New(ctx, nil) 即可。

    Agents.md（指令注入）
        核心功能：自动将 Agents.md 文件的内容注入到模型的输入消息中，为 Agent 提供项目级的指令或背景信息。
        关键配置：通常涉及 Agents.md 文件的路径。



In [ ]:
// 多个中间件组合
agent, err := adk.NewChatModelAgent(ctx, &adk.ChatModelAgentConfig{
    Model:       cm,
    Instruction: "...",
    Middlewares: []adk.ChatModelAgentMiddleware{
        patchMW,      // 1. 先修复悬空工具调用
        summaryMW,    // 2. 上下文压缩
        reductionMW,  // 3. 工具结果管理（若不用 FileSystem 的转存）
        fsMW,         // 4. 文件系统能力
        plantaskMW,   // 5. 任务规划
    },
})